In [ ]:
import os, json, time, shutil, hashlib, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from itertools import product

In [ ]:
TASK = Path.cwd().parent
PROFILE = os.environ.get('NEURAL_DNA_PROFILE', 'local')
INPUT, OUTPUT = TASK / 'input' / PROFILE, TASK / 'output' / PROFILE
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
cfg = json.loads((INPUT / 'config.json').read_text())
SEED = cfg['seed']
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(cfg['threads'])

In [ ]:
for cell in json.loads((INPUT / 'helpers.ipynb').read_text())['cells']:
    exec(''.join(cell['source']), globals())
carry(INPUT, OUTPUT)

In [ ]:
queries = pd.read_csv(INPUT / 'queries.csv')
values = np.load(INPUT / 'query_fingerprints.npz')['values']
truth = np.load(INPUT / 'truth.npz')['ancestry']
with np.load(INPUT / 'source_fingerprints.npz') as data:
    panels = {key: data[key] for key in data.files}
centers = panels['proxy'].mean(axis=1)
scale = np.maximum(np.mean((centers[0] - centers[1]) ** 2, axis=1), 1e-12)
np.save(OUTPUT / 'block_scale.npy', scale)

In [ ]:
toy = np.array([[0.1, 0.8], [0.4, 0.3], [0.8, 0.1], [0.7, 0.2]])
paths = np.array(list(product(range(2), repeat=4)))
energy = toy[np.arange(4)[None], paths].sum(axis=1) + 0.5 * (paths[:, 1:] != paths[:, :-1]).sum(axis=1)
assert np.array_equal(viterbi(toy, 0.5), paths[energy.argmin()])
probability = softmax(-energy / 0.3)
brute = np.array([[(probability * (paths[:, j] == k)).sum() for k in range(2)] for j in range(4)])
assert np.allclose(posterior(toy, 0.5, 0.3), brute, atol=1e-12)
assert np.allclose(source_probabilities(posterior(np.repeat(toy, 2, axis=1), 0, 0.3)), posterior(toy, 0, 0.3))
(OUTPUT / 'decoder_checks.json').write_text(json.dumps({'viterbi_exhaustive': True,
    'posterior_max_error': float(np.abs(posterior(toy, 0.5, 0.3) - brute).max()), 'source_aggregation': True}, indent=2))

In [ ]:
fits, grid = [], []
for panel in ['exact', 'proxy']:
    references = panels['exact_cal'] if panel == 'exact' else panels['proxy']
    for markers in cfg['marker_counts']:
        for steps in cfg['finetune_steps']:
            calibration = queries.query("split == 'cal' and steps == @steps")
            best = None
            for lam, temperature in product(cfg['lambda_grid'], cfg['temperature_grid']):
                losses = []
                for q in calibration.query_id:
                    cost = copying_cost(values[q], references, markers, scale)
                    probability = source_probabilities(posterior(cost, lam, temperature))
                    losses.append(-np.log(np.maximum(probability[np.arange(len(cost)), truth[q]], 1e-300)).mean())
                row = {'panel': panel, 'markers': markers, 'steps': steps, 'lambda': lam,
                    'temperature': temperature, 'calibration_nll': float(np.mean(losses))}
                grid.append(row)
                if best is None or row['calibration_nll'] < best['calibration_nll'] - 1e-12:
                    best = row.copy()
            fits.append(best)
pd.DataFrame(fits).to_csv(OUTPUT / 'fitted_parameters.csv', index=False)
pd.DataFrame(grid).to_csv(OUTPUT / 'calibration_grid.csv', index=False)
display(pd.DataFrame(fits))